# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


In [1]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
#import datetime
import os
from datetime import datetime
from pathlib import Path

In [2]:
class Exposure:
    
    def __init__(self, filename):

        self.filename = filename
        with fits.open(filename) as hdu_list:
            primary_header = hdu_list[0].header
            self.dpr_catg = self.getKeyVal(primary_header, 'HIERARCH ESO DPR CATG')
            self.dpr_type = self.getKeyVal(primary_header, 'HIERARCH ESO DPR TYPE')
            self.obsid = self.getKeyVal(primary_header, 'HIERARCH ESO OBS ID')
            self.OBstart = self.getKeyVal(primary_header, 'HIERARCH ESO OBS START')
            self.dateobs = self.getKeyVal(primary_header, 'DATE-OBS')
            self.tplnexp = self.getKeyVal(primary_header, 'HIERARCH ESO TPL NEXP')
            self.tplexpno = self.getKeyVal(primary_header, 'HIERARCH ESO TPL EXPNO')
            self.filter = self.getKeyVal(primary_header, 'HIERARCH ESO INS FILT1 NAME')
            self.exptime = self.getKeyVal(primary_header, 'EXPTIME')

            if (self.dpr_catg == 'SCIENCE' and self.dpr_type == 'OBJECT') or \
                (self.dpr_catg == 'CALIB' and self.dpr_type == 'STD,ZEROPOINT'):
                self.ra = self.getKeyVal(primary_header, 'RA')
                self.dec = self.getKeyVal(primary_header, 'DEC')
                self.tel_alpha = self.getKeyVal(primary_header, 'HIERARCH ESO TEL TARG ALPHA')
                self.tel_delta = self.getKeyVal(primary_header, 'HIERARCH ESO TEL TARG DELTA')
                self.target = self.getKeyVal(primary_header, 'HIERARCH ESO OBS TARG NAME').replace(' ', '_')
        
                # we add the FLI
                yyyy = int(self.dateobs[0:4])
                mm = int(self.dateobs[5:7])
                dd = int(self.dateobs[8:10])
                fli = self.get_phase_on_day(yyyy,mm,dd)
                if fli <= 0.5:
                    fli = fli*2.0
                else:
                    fli = 2.0*(1-fli)
                self.fli = fli  

    def get_phase_on_day(self, year, month, day):
        """Returns a floating-point number from 0-1. where 0=new, 0.5=full, 1=new"""
        #Ephem stores its date numbers as floating points, which the following uses
        #to conveniently extract the percent time between one new moon and the next
        #This corresponds (somewhat roughly) to the phase of the moon.

        #Use Year, Month, Day as arguments
        date=ephem.Date(datetime.date(year,month,day))

        nnm = ephem.next_new_moon    (date)
        pnm = ephem.previous_new_moon(date)

        lunation=(date-pnm)/(nnm-pnm)

        #Note that there is a ephem.Moon().phase() command, but this returns the
        #percentage of the moon which is illuminated. This is not really what we want.

        return lunation

            
    def getKeyVal(self,primary_header, key):
        val = primary_header[key]
        return val


class ExposuresPartition:
    
    def __init__(self, filename, frametype):

        #print('ExposurePartition  filename: '+filename+'\nExposurePartition frametype: '+frametype, end='\r')
        
        self.base_path = os.path.dirname(filename)
        self.frametype = frametype

        self.filter = None
        if frametype != 'BIAS' or frametype != 'BPM':
            self.filter = self.getFilter(filename)
        self.framelist = filename
        self.frame = None
        
        # for the moment the BPM have no time stamp
        self.OBstart = None
        if frametype != 'BPM':
            self.OBstart = self.getOBstart(filename)
            
        # we construct the names according to file type
        self.framelist = None
        self.frame = None
        if frametype == 'BPM':
            self.framelist = None
            self.frame = self.base_path+'/OMEGACAM_BPM_01.fits'
        elif frametype == 'BIAS':
            self.framelist = self.base_path+'/bias.list'
            self.frame = self.base_path+'/OMEGACAM_MB_1x1.fits'
        elif frametype == 'FLAT,DOME':
            self.framelist = self.base_path+'/dome.list'
            self.frame = self.base_path+'/OMEGACAM_DOMEFF_'+self.filter+'_1x1.fits'
        elif frametype == 'FLAT,SKY':
            self.framelist = self.base_path+'/sky.list'
            self.frame = self.base_path+'/OMEGACAM_SKYFF_'+self.filter+'_1x1.fits'
        elif frametype == 'MFF':
            self.framelist = self.base_path+'/mff.list'
            self.frame = self.base_path+'/OMEGACAM_MFF_'+self.filter+'_1x1.fits'
        elif frametype == 'SCIENCE' or frametype == 'SKY':
            self.framelist = self.base_path+'/science.list'            
            self.frame = None


        self.bpmPartition = None
        self.bpmFrame = None
        self.bpm_have = False
        self.bpm_have_link = False

        self.biasPartition = None
        self.biasFrame = None
        self.bias_have = False
        self.bias_have_link = False

        self.domePartition = None
        self.domeFrame = None

        self.skyPartition = None
        self.SkyFrame = None

        self.mffPartition = None
        self.mffFrame = None
        
    def insertExposure(self, exposure):

        # we select the partition according to exposure dpr.type
        if 'BIAS' in exposure.dpr_type:
            key_str = 'bias/'+'%d'%exposure.obsid+'/'+exposure.OBstart
        elif 'FLAT,DOME' in exposure.dpr_type:
            key_str = 'dome/'+'%d'%exposure.obsid+'/'+exposure.filter+'/'+exposure.OBstart
        elif 'FLAT,SKY' in exposure.dpr_type:
            key_str = 'sky/'+'%d'%exposure.obsid+'/'+exposure.filter+'/'+exposure.OBstart
        elif ('CALIB' in exposure.dpr_catg) and ('STD,ZEROPOINT' in exposure.dpr_type):
            key_str = 'std/'+exposure.target + '_' + \
                    exposure.filter+'/'+\
                    exposure.OBstart
        elif ('SCIENCE' in exposure.dpr_catg) and ('OBJECT' in exposure.dpr_type):
            key_str = 'science/'+exposure.target+'_'+\
                    ("%010.3f"%exposure.tel_alpha).replace('.','p')+'_'+\
                    ("%09.1f"%exposure.tel_delta).replace('.','p').replace('-','m')+'/'+\
                    exposure.filter+'/'+\
                    exposure.OBstart
        else:
            print('dpr type: %s'%exposure.dpr_type, ' not classifiable.', end='\r')
            key_str = 'unknown/'+'%s'%exposure.dpr_type


        self.base_path = key_str
        self.exposureList.append(exposure)
        self.OBstart = exposure.OBstart
        self.filter = exposure.filter
        if key_str in self.partList:
            self.partList[key_str].append(exposure)
        else:
            self.partList[key_str] = [exposure]
            self.OBstartDict[key_str] = exposure.OBstart
            self.filterDict[key_str] = exposure.filter
        
    def getFilter(self,filename):
        fields = filename.split('/')
        return fields[-3]
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]

def timeDistance(time_str1, time_str2):
    FMT = '%Y-%m-%dT%H:%M:%S'
    return abs((datetime.strptime(time_str1, FMT) - datetime.strptime(time_str2, FMT)).total_seconds())

def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val

class SciencePartition:
    
    def __init__(self, filename):

        self.base_path = os.path.dirname(filename)
        self.filter = self.getFilter(filename)
        self.OBstart = self.getOBstart(filename)
        self.bpmPartition = None
        self.biasPartition = None
        self.biasFrame = None
        self.domePartition = None
        self.domeFrame = None
        self.skyPartition = None
        self.SkyFrame = None
        self.mffPartition = None
        self.mffFrame = None
        
    def getFilter(self,filename):
        fields = filename.split('/')
        return fields[-3]
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]
    
class BiasPartition:
    
    def __init__(self, filename):

        self.base_path = os.path.dirname(filename)
        self.OBstart = self.getOBstart(filename)
        self.bpmPartition = None
        #self.biasPartition = None
        self.mffPartition = None
        
    def getOBstart(self, filename):
        fields = filename.split('/')
        return fields[-2]    

def timeDistance(time_str1, time_str2):
    FMT = '%Y-%m-%dT%H:%M:%S'
    return abs((datetime.strptime(time_str1, FMT) - datetime.strptime(time_str2, FMT)).total_seconds())

def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val

In [3]:
# we set the data reduction directories
raw_data_dir = '/home/lschmidt/fernando/fits/'
reduced_cal_dir = '/home/fselman/omegacam/fernando/reduced/calibs/'
reduced_science_dir = '/home/fselman/omegacam/fernando/reduced/science/'

In [4]:
bpm_list = glob.glob(reduced_cal_dir+'**/bpm.list', recursive=True)
print('number of bpm partitions: ', len(bpm_list))
bpm_partitions = []
for bpm_file in bpm_list:
    bpm_partitions.append(BiasPartition(bpm_file))
    
for part in bpm_partitions:
    print(part.OBstart)

number of bpm partitions:  1
bpm


In [5]:
bias_list = glob.glob(reduced_cal_dir+'**/bias.list', recursive=True)
print('number of bias partitions: ', len(bias_list))
bias_partitions = []
for bias_file in bias_list:
    bias_partitions.append(BiasPartition(bias_file))
    
#for part in bias_partitions:
#    print(part.OBstart)

number of bias partitions:  67


In [6]:
dome_list = glob.glob(reduced_cal_dir+'**/dome.list', recursive=True)
print('number of dome partitions: ', len(dome_list))
dome_partitions = []
for dome_file in dome_list:
    #print(dome_file)
    dome_partitions.append(SciencePartition(dome_file))
    
#for part in dome_partitions:
#    print(part.OBstart, part.filter)

number of dome partitions:  33


In [7]:
sky_list = glob.glob(reduced_cal_dir+'**/sky.list', recursive=True)
print('number of sky partitions: ', len(sky_list))
sky_partitions = []
for sky_file in sky_list:
    #print(sky_file)
    sky_partitions.append(SciencePartition(sky_file))
    
#for part in sky_partitions:
#    print(part.OBstart, part.filter)

number of sky partitions:  33


In [8]:
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

sci_partitions = []
for science_file in science_list:
    sci_partitions.append(SciencePartition(science_file))
    
#for part in sci_partitions:
#    print(part.OBstart, part.filter)



number of science partitions:  205


In [9]:
#We associate a bpm partition to each science frame
for sci_part in sci_partitions:
    sci_part.bpmPartition = bpm_partitions[0]
    #print((sci_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (sci_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sci_part.bpmPartition).base_path+'/toreduce_sci').touch()
        print(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))


/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_01.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08/OMEGACAM_BPM_01.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_02.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08/OMEGACAM_BPM_02.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_03.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08/OMEGACAM_BPM_03.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08/OMEGACAM_BPM_04.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08/OMEGACAM_BPM_05.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_14.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41/OMEGACAM_BPM_14.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_15.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41/OMEGACAM_BPM_15.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_16.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41/OMEGACAM_BPM_16.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_17.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41/OMEGACAM_BPM_17.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_18.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41/OMEGACAM_BPM_18.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_25.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41/OMEGACAM_BPM_25.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_26.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41/OMEGACAM_BPM_26.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_27.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41/OMEGACAM_BPM_27.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_28.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41/OMEGACAM_BPM_28.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_29.fits /home/fselman/omegacam/fernando/reduced/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41/OMEGACAM_BPM_29.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T03:26:06/OMEGACAM_BPM_04.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T03:26:06/OMEGACAM_BPM_05.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_06.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T03:26:06/OMEGACAM_BPM_06.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_07.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T03:26:06/OMEGACAM_BPM_07.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_08.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T03:26:06/OMEGACAM_BPM_08.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_15.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T00:19:52/OMEGACAM_BPM_15.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_16.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T00:19:52/OMEGACAM_BPM_16.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_17.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T00:19:52/OMEGACAM_BPM_17.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_18.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T00:19:52/OMEGACAM_BPM_18.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_19.fits /home/fselman/omegacam/fernando/reduced/science/IX_Vel_081518p989_m491319p8/NB_659/2017-03-11T00:19:52/OMEGACAM_BPM_19.fits
/home/fselman/omegac

...

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_03.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T05:46:37/OMEGACAM_BPM_03.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T05:46:37/OMEGACAM_BPM_04.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T05:46:37/OMEGACAM_BPM_05.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_06.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T05:46:37/OMEGACAM_BPM_06.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_07.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T05:46:37/OMEGACAM_BPM_07.fits
/home/fsel

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_16.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T07:30:31/OMEGACAM_BPM_16.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_17.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T07:30:31/OMEGACAM_BPM_17.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_18.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T07:30:31/OMEGACAM_BPM_18.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_19.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T07:30:31/OMEGACAM_BPM_19.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_20.fits /home/fselman/omegacam/fernando/reduced/science/V584_Lyr_191059p420_0285638p9/r_SDSS/2017-06-16T07:30:31/OMEGACAM_BPM_20.fits
/home/fsel

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_28.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/NB_659/2017-07-05T06:08:34/OMEGACAM_BPM_28.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_29.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/NB_659/2017-07-05T06:08:34/OMEGACAM_BPM_29.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_30.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/NB_659/2017-07-05T06:08:34/OMEGACAM_BPM_30.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_31.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/NB_659/2017-07-05T06:08:34/OMEGACAM_BPM_31.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_32.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/NB_659/2017-07-05T06:08:34/OMEGACAM_BPM_32.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_10.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_BPM_10.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_11.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_BPM_11.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_12.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_BPM_12.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_13.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_BPM_13.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_14.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_BPM_14.fits
/home/fselman/omegac

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_23.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T08:06:04/OMEGACAM_BPM_23.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_24.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T08:06:04/OMEGACAM_BPM_24.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_25.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T08:06:04/OMEGACAM_BPM_25.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_26.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T08:06:04/OMEGACAM_BPM_26.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_27.fits /home/fselman/omegacam/fernando/reduced/science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T08:06:04/OMEGACAM_BPM_27.fits
/home/fselman/omegac

In [10]:

# we associate a bias partition to each science partition

for sci_part in sci_partitions:
    # here we look for the closest in time bias partition
    sci_date = sci_part.OBstart

    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sci_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sci_part.biasPartition = bias_part
    
    # we find the sky closest in time and we link the mff to a file that
    # will be present there once it is created
    #
    skyBestDeltaT = 3600.*24.*30.0
    for sky_part in sky_partitions:
        if sci_part.filter != sky_part.filter:
            continue
        sky_date = sky_part.OBstart
        deltaT = timeDistance(sci_date, sky_date)

        if deltaT < skyBestDeltaT:
            skyBestDeltaT = deltaT
            sci_part.skyPartition = sky_part
            sci_part.mffPartition = sky_part

    
    print('science: %s %s'%(sci_part.filter, sci_part.OBstart,)+\
            '     bias: %s (%3.1f)'%(sci_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' sky: %s %s (%3.1f)'%(sci_part.skyPartition.filter, sci_part.skyPartition.OBstart, skyBestDeltaT/3600.0/24.,))

# at this point we have the biases and mff for each science frame. Now we link
for sci_part in sci_partitions:
    mbias = (sci_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sci_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sci_part.biasPartition).base_path+'/toreduce_sci').touch()
        print(mbias)
        
#    skyff = (sci_part.skyPartition).base_path+'/OMEGACAM_SKYFF_'+sci_part.filter+'_1x1.fits'
#    if not Path(skyff).is_file():
#        try:
#            os.symlink(skyff, sci_part.base_path+'/OMEGACAM_SKYFF_'+sci_part.filter+'_1x1.fits')
#        except:
#            pass
#        Path((sci_part.skyPartition).base_path+'/toreduce_sci').touch() 

    mff = (sci_part.mffPartition).base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits'
    if not Path(mff).is_file():
        try:
            os.symlink(mff, sci_part.base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits')
        except:
            pass
        Path((sci_part.mffPartition).base_path+'/toreduce_mff').touch()


science: r_SDSS 2017-03-11T01:24:08     bias: 2017-03-11T09:21:33 (0.3) sky: r_SDSS 2017-03-13T23:12:13 (2.9)
science: r_SDSS 2017-03-12T00:24:41     bias: 2017-03-12T09:32:25 (0.4) sky: r_SDSS 2017-03-13T23:12:13 (1.9)
science: NB_659 2017-03-12T00:24:41     bias: 2017-03-12T09:32:25 (0.4) sky: NB_659 2017-03-08T23:14:31 (3.0)
science: NB_659 2017-03-11T01:24:08     bias: 2017-03-11T09:21:33 (0.3) sky: NB_659 2017-03-08T23:14:31 (2.1)
science: NB_659 2017-03-11T03:26:06     bias: 2017-03-11T09:21:33 (0.2) sky: NB_659 2017-03-08T23:14:31 (2.2)
science: NB_659 2017-03-11T00:19:52     bias: 2017-03-11T09:21:33 (0.4) sky: NB_659 2017-03-08T23:14:31 (2.0)
science: r_SDSS 2017-03-11T03:26:06     bias: 2017-03-11T09:21:33 (0.2) sky: r_SDSS 2017-03-13T23:12:13 (2.8)
science: r_SDSS 2017-03-11T04:32:08     bias: 2017-03-11T09:21:33 (0.2) sky: r_SDSS 2017-03-13T23:12:13 (2.8)
science: r_SDSS 2017-03-11T04:47:04     bias: 2017-03-11T09:21:33 (0.2) sky: r_SDSS 2017-03-13T23:12:13 (2.8)
science: r

/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-12T09:32:25/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-12T09:32:25/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-03-11T09:21:33/OMEGACAM_MB_1x1.fits


/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-09T09:56:01/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-09T09:56:01/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-09T09:56:01/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-09T09:56:01/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-03T11:18:07/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-10T10:21:09/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-03T11:18:07/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-10T10:21:09/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-10T10:21:09/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-06-18T10:17:21/OMEGACAM_MB_1x1.fits


/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-16T09:49:33/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-03T10:17:56/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-18T09:39:29/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-13T09:54:14/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-13T09:54:14/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-13T09:54:14/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-18T09:39:29/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-05-03T10:17:56/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-15T10:03:38/OMEGACAM_MB_1x1.fits
/home/fselman/omegacam/fernando/reduced/calibs/bias/574184/2017-04-15T10:03:38/OMEGACAM_MB_1x1.fits


In [11]:
#We associate a bpm partition to each sky frame partition
for sky_part in sky_partitions:
    sky_part.bpmPartition = bpm_partitions[0]
    #print((sky_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (sky_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sky_part.bpmPartition).base_path+'/toreduce_sky').touch()
        print(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_01.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_01.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_02.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_02.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_03.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_03.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_04.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_05.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_06.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_20.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-17T23:16:17/OMEGACAM_BPM_20.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_21.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-17T23:16:17/OMEGACAM_BPM_21.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_22.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-17T23:16:17/OMEGACAM_BPM_22.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_23.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-17T23:16:17/OMEGACAM_BPM_23.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_24.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-17T23:16:17/OMEGACAM_BPM_24.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_25.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_08.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-30T22:57:09/OMEGACAM_BPM_08.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_09.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-30T22:57:09/OMEGACAM_BPM_09.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_10.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-30T22:57:09/OMEGACAM_BPM_10.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_11.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-30T22:57:09/OMEGACAM_BPM_11.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_12.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-03-30T22:57:09/OMEGACAM_BPM_12.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_13.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_23.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-04-05T22:53:48/OMEGACAM_BPM_23.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_24.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-04-05T22:53:48/OMEGACAM_BPM_24.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_25.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-04-05T22:53:48/OMEGACAM_BPM_25.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_26.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-04-05T22:53:48/OMEGACAM_BPM_26.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_27.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-04-05T22:53:48/OMEGACAM_BPM_27.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_28.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-05-29T22:17:40/OMEGACAM_BPM_05.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_06.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-05-29T22:17:40/OMEGACAM_BPM_06.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_07.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-05-29T22:17:40/OMEGACAM_BPM_07.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_08.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-05-29T22:17:40/OMEGACAM_BPM_08.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_09.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253584/r_SDSS/2017-05-29T22:17:40/OMEGACAM_BPM_09.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_10.fits /home/fselman/omegacam/

...

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_22.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-03-17T23:05:06/OMEGACAM_BPM_22.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_23.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-03-17T23:05:06/OMEGACAM_BPM_23.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_24.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-03-17T23:05:06/OMEGACAM_BPM_24.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_25.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-03-17T23:05:06/OMEGACAM_BPM_25.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_26.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-03-17T23:05:06/OMEGACAM_BPM_26.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_27.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_06.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-04-16T22:37:00/OMEGACAM_BPM_06.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_07.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-04-16T22:37:00/OMEGACAM_BPM_07.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_08.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-04-16T22:37:00/OMEGACAM_BPM_08.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_09.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-04-16T22:37:00/OMEGACAM_BPM_09.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_10.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-04-16T22:37:00/OMEGACAM_BPM_10.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_11.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_26.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-03T22:16:59/OMEGACAM_BPM_26.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_27.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-03T22:16:59/OMEGACAM_BPM_27.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_28.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-03T22:16:59/OMEGACAM_BPM_28.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_29.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-03T22:16:59/OMEGACAM_BPM_29.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_30.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-03T22:16:59/OMEGACAM_BPM_30.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_31.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_08.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-31T22:14:10/OMEGACAM_BPM_08.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_09.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-31T22:14:10/OMEGACAM_BPM_09.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_10.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-31T22:14:10/OMEGACAM_BPM_10.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_11.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-31T22:14:10/OMEGACAM_BPM_11.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_12.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-05-31T22:14:10/OMEGACAM_BPM_12.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_13.fits /home/fselman/omegacam/

/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_29.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-06-09T22:11:32/OMEGACAM_BPM_29.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_30.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-06-09T22:11:32/OMEGACAM_BPM_30.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_31.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-06-09T22:11:32/OMEGACAM_BPM_31.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_32.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-06-09T22:11:32/OMEGACAM_BPM_32.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_01.fits /home/fselman/omegacam/fernando/reduced/calibs/sky/200253598/NB_659/2017-06-19T22:15:25/OMEGACAM_BPM_01.fits
/home/fselman/omegacam/fernando/reduced/calibs/bpm/OMEGACAM_BPM_02.fits /home/fselman/omegacam/

In [12]:
# here we associate dome and bias to each sky partition
for sky_part in sky_partitions:
    sky_date = sky_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sky_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sky_part.biasPartition = bias_part
    #print(biasBestDeltaT/3600.0/24.0)
    
    # we find the dome closest in time and we link the domeff to a file that
    # will be present there once it is created
    #
    domeBestDeltaT = 3600.*24.*30.0
    for dome_part in dome_partitions:
        if sky_part.filter == dome_part.filter:
            dome_date = dome_part.OBstart
            deltaT = timeDistance(sky_date, dome_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < domeBestDeltaT:
                domeBestDeltaT = deltaT
                sky_part.domePartition = dome_part
        else:
            pass
    #print(domeBestDeltaT/3600.0/24.0)
    
    print('sky: %s %s'%(sky_part.filter, sky_part.OBstart,)+\
            '     bias: %s (%4.2f)'%(sky_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' dome: %s %s (%4.2f)'%(sky_part.domePartition.filter, sky_part.domePartition.OBstart, domeBestDeltaT/3600.0/24.,))
    
# at this point we have the biases and domes for each sky ff frame. Now we link
for sky_part in sky_partitions:
    mbias = (sky_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sky_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sky_part.biasPartition).base_path+'/toreduce_sky').touch()
        #print(mbias)
        
    domeff = (sky_part.domePartition).base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits'
    if not Path(domeff).is_file():
        try:
            os.symlink(domeff, sky_part.base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits')
        except:
            pass
        Path((sky_part.domePartition).base_path+'/toreduce_sky').touch()
        #print(domeff)    

sky: r_SDSS 2017-03-13T23:12:13     bias: 2017-03-14T09:41:34 (0.44) dome: r_SDSS 2017-03-14T09:41:34 (0.44)
sky: r_SDSS 2017-03-17T23:16:17     bias: 2017-03-18T10:11:14 (0.45) dome: r_SDSS 2017-03-18T16:03:36 (0.70)
sky: r_SDSS 2017-03-25T23:11:55     bias: 2017-03-26T10:26:41 (0.47) dome: r_SDSS 2017-03-24T11:58:39 (1.47)
sky: r_SDSS 2017-03-30T22:57:09     bias: 2017-03-30T21:38:06 (0.05) dome: r_SDSS 2017-03-30T09:44:01 (0.55)
sky: r_SDSS 2017-04-05T22:53:48     bias: 2017-04-06T09:52:40 (0.46) dome: r_SDSS 2017-04-05T10:02:10 (0.54)
sky: r_SDSS 2017-05-07T22:16:25     bias: 2017-05-08T10:04:26 (0.49) dome: r_SDSS 2017-05-08T10:04:26 (0.49)
sky: r_SDSS 2017-05-29T22:17:40     bias: 2017-05-29T10:26:43 (0.49) dome: r_SDSS 2017-05-30T10:14:23 (0.50)
sky: r_SDSS 2017-06-09T22:17:03     bias: 2017-06-10T10:21:09 (0.50) dome: r_SDSS 2017-06-09T10:10:02 (0.50)
sky: r_SDSS 2017-06-18T22:17:32     bias: 2017-06-18T10:17:21 (0.50) dome: r_SDSS 2017-06-18T10:17:21 (0.50)
sky: r_SDSS 2017-06

In [13]:
#We associate a bpm partition to each dome ff frame
for dome_part in dome_partitions:
    dome_part.bpmPartition = bpm_partitions[0]
    #print((dome_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (dome_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, dome_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((dome_part.bpmPartition).base_path+'/toreduce_dome').touch()
        #print(bpm_file, dome_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))

    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part        
    
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        #print(mbias)

In [14]:
# here we associate biases to each dome partition
for dome_part in dome_partitions:
    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part
            #print(dome_date, bias_date, biasBestDeltaT/3600.0/24.0)

    
    print('dome: %s %s'%(dome_part.filter, dome_part.OBstart,)+\
            '     bias: %s (%5.3f)'%((dome_part.biasPartition).OBstart, biasBestDeltaT/3600.0/24.,))
    
# at this point we have the biases  for each dome ff frame. Now we link
for dome_part in dome_partitions:
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        #print(mbias)
        

dome: r_SDSS 2017-04-13T18:36:09     bias: 2017-04-13T09:54:14 (0.362)
dome: r_SDSS 2017-03-14T09:41:34     bias: 2017-03-14T09:41:34 (0.000)
dome: r_SDSS 2017-03-18T16:03:36     bias: 2017-03-18T10:11:14 (0.245)
dome: r_SDSS 2017-03-24T11:58:39     bias: 2017-03-24T10:02:08 (0.081)
dome: r_SDSS 2017-04-05T10:02:10     bias: 2017-04-05T10:02:10 (0.000)
dome: r_SDSS 2017-06-18T10:17:21     bias: 2017-06-18T10:17:21 (0.000)
dome: r_SDSS 2017-06-27T15:07:22     bias: 2017-06-27T09:05:42 (0.251)
dome: r_SDSS 2017-07-06T10:20:27     bias: 2017-07-06T10:20:27 (0.000)
dome: r_SDSS 2017-03-08T09:39:09     bias: 2017-03-08T09:39:09 (0.000)
dome: r_SDSS 2017-03-30T09:44:01     bias: 2017-03-30T09:44:01 (0.000)
dome: r_SDSS 2017-04-17T09:58:07     bias: 2017-04-17T09:58:07 (0.000)
dome: r_SDSS 2017-05-02T10:04:48     bias: 2017-05-02T10:04:48 (0.000)
dome: r_SDSS 2017-05-05T09:59:05     bias: 2017-05-05T09:59:05 (0.000)
dome: r_SDSS 2017-05-08T10:04:26     bias: 2017-05-08T10:04:26 (0.000)
dome: 

In [15]:
##Finalle associate a bpm partition to each bias frame
for bias_part in bias_partitions:
    bias_part.bpmPartition = bpm_partitions[0]
    #print((bias_part.bpmPartition).base_path)
    for i in range(1,33):
        bpm_file = (bias_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, bias_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((bias_part.bpmPartition).base_path+'/toreduce_bias').touch()
        #print(bpm_file, bias_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))